# KKR Performance Analysis: IPL 2023–2024

## 1. Objective

Analyse Kolkata Knight Riders' performance across IPL 2023 and IPL 2024
to identify team performance trends, player contributions, and
potential tactical insights.

## 2. Analytical Scope

- Team-level performance and match outcomes
- Batting contributions and scoring efficiency
- Bowling contributions and effectiveness
- Performance across Powerplay, Middle, and Death phases
- Evidence-based scouting and tactical recommendations

## 3. Data Limitations

- The source dataset contains delivery-level records.
- KKR's final two league matches in IPL 2024 are absent.
- IPL 2024 Qualifier 1 and Final are included.
- Previously identified bowling-run anomalies remain unresolved.

Conclusions must account for data coverage, player roles, and the
limitations of the available metrics. Observed associations must not
be presented as proof of causation.

In [7]:
from pathlib import Path
import sys
import pandas as pd

current_path = Path.cwd()

project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "src" / "config.py").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing src/config.py."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print("Project root successfully added to the Python path.")

Project root: /Users/soham/MyDocs/Projects/Cricket Analysis/Hackathon - DPL 2026/CodeFiles/Hackathon_DPL2026
Project root successfully added to the Python path.


In [3]:
from src.data_loader import load_raw_data
from src.features.metric_definitions import add_metric_features
from src.features.match_level import (
    build_match_summary,
    build_team_match,
)
from src.features.player_level import (
    build_player_season_batting,
    build_player_season_bowling,
    build_player_season_phase_batting,
    build_player_season_phase_bowling,
)

print("All required project modules imported successfully.")

All required project modules imported successfully.


In [4]:
df = load_raw_data()

print(f"Raw dataset shape: {df.shape}")
display(df.head())

Raw dataset shape: (34967, 61)


,row_id,p_match,inns,bat,p_bat,team_bat,bowl,p_bowl,team_bowl,ball,...,wagonzone,line,length,shot,control,predscore,wprob,bowl_speed_category,bowl_release_point,bowl_type
0,225793,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,1,...,6,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,162,49.42,3.0,2.0,OUTSWINGER
1,225794,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,2,...,0,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,FLICK,0.0,161,48.27,3.0,2.0,OUTSWINGER
2,225795,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,3,...,5,OUTSIDE_OFFSTUMP,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,160,47.12,2.0,2.0,ANGLED IN
3,225796,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,4,...,4,ON_THE_STUMPS,GOOD_LENGTH,ON_DRIVE,1.0,160,47.12,2.0,2.0,OUTSWINGER
4,225797,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,5,...,5,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,158,44.82,3.0,2.0,INSWINGER


In [8]:
# Apply the validated metric definitions
df_features = add_metric_features(df)

print("Raw dataset shape:", df.shape)
print("Feature dataset shape:", df_features.shape)

new_columns = sorted(set(df_features.columns) - set(df.columns))
print("\nDerived metric columns:", new_columns)

Raw dataset shape: (34967, 61)
Feature dataset shape: (34967, 68)

Derived metric columns: ['batting_dot', 'boundary_ball', 'boundary_runs', 'bowler_credited_wicket', 'bowling_dot', 'legal_delivery', 'match_phase']


In [9]:
# Match-level datasets
match_summary = build_match_summary(df_features)
team_match = build_team_match(df_features)

# Player-level datasets
player_season_batting = build_player_season_batting(df_features)
player_season_bowling = build_player_season_bowling(df_features)

player_season_phase_batting = build_player_season_phase_batting(df_features)
player_season_phase_bowling = build_player_season_phase_bowling(df_features)

# Display dataset sizes
dataset_summary = pd.DataFrame([
    {"dataset": "match_summary", "rows": len(match_summary)},
    {"dataset": "team_match", "rows": len(team_match)},
    {"dataset": "player_season_batting", "rows": len(player_season_batting)},
    {"dataset": "player_season_bowling", "rows": len(player_season_bowling)},
    {"dataset": "player_season_phase_batting", "rows": len(player_season_phase_batting)},
    {"dataset": "player_season_phase_bowling", "rows": len(player_season_phase_bowling)},
])

display(dataset_summary)

,dataset,rows
0,match_summary,145
1,team_match,290
2,player_season_batting,347
3,player_season_bowling,273
4,player_season_phase_batting,733
5,player_season_phase_bowling,672


In [10]:
assert df_features.shape[0] == df.shape[0]
assert len(match_summary) == 145
assert len(team_match) == 290

assert not player_season_batting.duplicated(["p_bat", "year"]).any()
assert not player_season_bowling.duplicated(["p_bowl", "year"]).any()

assert not player_season_phase_batting.duplicated(
    ["p_bat", "year", "match_phase"]
).any()

assert not player_season_phase_bowling.duplicated(
    ["p_bowl", "year", "match_phase"]
).any()

print("All analytical input checks passed.")

All analytical input checks passed.


In [11]:
# Filter KKR's team-match records
kkr_matches = team_match.loc[
    team_match["team"].eq("Kolkata Knight Riders")
].copy()

print("KKR team-match records:", len(kkr_matches))

display(
    kkr_matches.sort_values(["year", "date"])
    if "date" in kkr_matches.columns
    else kkr_matches.sort_values(["year", "p_match"])
)

KKR team-match records: 28


,p_match,team,opponent,winner,team_result,year
2,1359476,Kolkata Knight Riders,Punjab Kings,Punjab Kings,loss,2023
16,1359483,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,win,2023
25,1359487,Kolkata Knight Riders,Gujarat Titans,Kolkata Knight Riders,win,2023
36,1359493,Kolkata Knight Riders,Sunrisers Hyderabad,Sunrisers Hyderabad,loss,2023
42,1359496,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians,loss,2023
55,1359502,Kolkata Knight Riders,Delhi Capitals,Delhi Capitals,loss,2023
65,1359507,Kolkata Knight Riders,Chennai Super Kings,Chennai Super Kings,loss,2023
70,1359510,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,win,2023
77,1359513,Kolkata Knight Riders,Gujarat Titans,Gujarat Titans,loss,2023
92,1359521,Kolkata Knight Riders,Sunrisers Hyderabad,Kolkata Knight Riders,win,2023


In [12]:
print("Available columns:")
print(kkr_matches.columns.tolist())

display(kkr_matches.head())

Available columns:
['p_match', 'team', 'opponent', 'winner', 'team_result', 'year']


,p_match,team,opponent,winner,team_result,year
2,1359476,Kolkata Knight Riders,Punjab Kings,Punjab Kings,loss,2023
16,1359483,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,win,2023
25,1359487,Kolkata Knight Riders,Gujarat Titans,Kolkata Knight Riders,win,2023
36,1359493,Kolkata Knight Riders,Sunrisers Hyderabad,Sunrisers Hyderabad,loss,2023
42,1359496,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians,loss,2023


In [14]:
# Summarise KKR results by season
kkr_results = (
    kkr_matches
    .groupby("year")
    .agg(
        matches_represented=("p_match", "nunique"),
        wins=("team_result", lambda s: s.eq("win").sum()),
        losses=("team_result", lambda s: s.eq("loss").sum()),
        unclassified=("team_result", lambda s: (~s.isin(["win", "loss"])).sum()),
    )
    .reset_index()
)

kkr_results["classified_matches"] = (
    kkr_results["wins"] + kkr_results["losses"]
)

kkr_results["win_pct"] = (
    100 * kkr_results["wins"]
    / kkr_results["classified_matches"].replace(0, pd.NA)
)

display(kkr_results.round(2))

,year,matches_represented,wins,losses,unclassified,classified_matches,win_pct
0,2023,14,6,8,0,14,42.86
1,2024,14,11,3,0,14,78.57


In [15]:
print("KKR result classifications:")
display(
    kkr_matches
    .groupby(["year", "team_result"], dropna=False)
    .size()
    .rename("matches")
    .reset_index()
)

print("Total represented matches:", kkr_matches["p_match"].nunique())
print("Unique KKR team-match records:", len(kkr_matches))

assert not kkr_matches["p_match"].duplicated().any()
assert kkr_matches["team_result"].isin(
    ["win", "loss", "unclassified"]
).all()

print("Match outcome checks passed.")

KKR result classifications:


,year,team_result,matches
0,2023,loss,8
1,2023,win,6
2,2024,loss,3
3,2024,win,11


Total represented matches: 28
Unique KKR team-match records: 28
Match outcome checks passed.


In [17]:
print(
    df_features[
        ["p_match", "year", "inns", "team_bat", "score"]
    ].head()
)

   p_match  year  inns             team_bat  score
0  1359475  2023     1  Chennai Super Kings      0
1  1359475  2023     1  Chennai Super Kings      1
2  1359475  2023     1  Chennai Super Kings      0
3  1359475  2023     1  Chennai Super Kings      1
4  1359475  2023     1  Chennai Super Kings      0


In [18]:
# Aggregate delivery scores to team-innings totals
team_innings = (
    df_features
    .groupby(
        ["p_match", "year", "inns", "team_bat"],
        as_index=False,
        dropna=False,
    )
    .agg(
        innings_score=("score", "sum"),
        recorded_deliveries=("score", "size"),
    )
)

# Keep KKR batting innings
kkr_innings = team_innings.loc[
    team_innings["team_bat"].eq("Kolkata Knight Riders")
].copy()

display(kkr_innings.head())
print("KKR innings records:", len(kkr_innings))

,p_match,year,inns,team_bat,innings_score,recorded_deliveries
3,1359476,2023,2,Kolkata Knight Riders,146,99
16,1359483,2023,1,Kolkata Knight Riders,204,129
25,1359487,2023,2,Kolkata Knight Riders,207,123
37,1359493,2023,2,Kolkata Knight Riders,204,128
42,1359496,2023,1,Kolkata Knight Riders,185,128


KKR innings records: 28


In [19]:
# Summarise KKR's innings scoring by season
kkr_scoring_summary = (
    kkr_innings
    .groupby("year")
    .agg(
        innings=("p_match", "size"),
        mean_score=("innings_score", "mean"),
        median_score=("innings_score", "median"),
        std_score=("innings_score", "std"),
        min_score=("innings_score", "min"),
        max_score=("innings_score", "max"),
        innings_180_plus=(
            "innings_score",
            lambda s: s.ge(180).sum(),
        ),
        innings_200_plus=(
            "innings_score",
            lambda s: s.ge(200).sum(),
        ),
    )
    .reset_index()
)

display(kkr_scoring_summary.round(2))

,year,innings,mean_score,median_score,std_score,min_score,max_score,innings_180_plus,innings_200_plus
0,2023,14,175.86,180.5,25.16,127,207,7,4
1,2024,14,190.14,177.5,47.01,114,272,7,6


In [20]:
display(
    kkr_innings[
        ["year", "p_match", "inns", "innings_score"]
    ]
    .sort_values(["year", "innings_score"], ascending=[True, False])
    .reset_index(drop=True)
)

,year,p_match,inns,innings_score
0,2023,1359487,2,207
1,2023,1359483,1,204
2,2023,1359493,2,204
3,2023,1359510,1,200
4,2023,1359507,2,186
5,2023,1359496,1,185
6,2023,1359527,2,182
7,2023,1359513,1,179
8,2023,1359542,2,175
9,2023,1359521,1,171


In [21]:
print(match_summary.columns.tolist())
display(match_summary.head())

['p_match', 'year', 'match_date', 'ground', 'country', 'winner', 'toss', 'represented_batting_teams', 'represented_bowling_teams', 'represented_teams', 'represented_innings', 'single_batting_team_flag', 'metadata_conflict_flag', 'result_class']


,p_match,year,match_date,ground,country,winner,toss,represented_batting_teams,represented_bowling_teams,represented_teams,represented_innings,single_batting_team_flag,metadata_conflict_flag,result_class
0,1359475,2023,3/31/2023,"Narendra Modi Stadium, Ahmedabad",India,Gujarat Titans,Gujarat Titans,2,2,2,2,False,False,classified
1,1359476,2023,4/1/2023,"Punjab Cricket Association IS Bindra Stadium, ...",India,Punjab Kings,Kolkata Knight Riders,2,2,2,2,False,False,classified
2,1359477,2023,4/1/2023,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,India,Lucknow Super Giants,Delhi Capitals,2,2,2,2,False,False,classified
3,1359478,2023,4/2/2023,"Rajiv Gandhi International Stadium, Uppal, Hyd...",India,Rajasthan Royals,Sunrisers Hyderabad,2,2,2,2,False,False,classified
4,1359479,2023,4/2/2023,"M Chinnaswamy Stadium, Bengaluru",India,Royal Challengers Bengaluru,Royal Challengers Bangalore,2,2,2,2,False,False,classified


In [22]:
# Count represented innings for each match
match_innings_coverage = (
    team_innings
    .groupby(["p_match", "year"])
    .agg(
        represented_innings=("inns", "nunique"),
        represented_batting_teams=("team_bat", "nunique"),
    )
    .reset_index()
)

display(
    match_innings_coverage
    .groupby(["year", "represented_innings"])
    .size()
    .rename("matches")
    .reset_index()
)

,year,represented_innings,matches
0,2023,1,1
1,2023,2,73
2,2024,2,71


In [23]:
incomplete_innings_coverage = match_innings_coverage.loc[
    match_innings_coverage["represented_innings"].ne(2)
]

display(incomplete_innings_coverage)

,p_match,year,represented_innings,represented_batting_teams
44,1359519,2023,1,1


In [24]:
# Retain matches with exactly two represented innings
complete_innings_matches = match_innings_coverage.loc[
    match_innings_coverage["represented_innings"].eq(2),
    ["p_match", "year"],
]

# Keep the innings totals from those matches
complete_team_innings = team_innings.merge(
    complete_innings_matches,
    on=["p_match", "year"],
    how="inner",
    validate="many_to_one",
)

print("Matches retained:", complete_innings_matches["p_match"].nunique())
print("Team-innings records retained:", len(complete_team_innings))

Matches retained: 144
Team-innings records retained: 288


In [25]:
# KKR's innings
kkr_innings_complete = complete_team_innings.loc[
    complete_team_innings["team_bat"].eq("Kolkata Knight Riders"),
    ["p_match", "year", "inns", "innings_score"],
].rename(
    columns={
        "inns": "kkr_innings_number",
        "innings_score": "kkr_score",
    }
)

# Opponents' innings
opponent_innings = complete_team_innings.loc[
    ~complete_team_innings["team_bat"].eq("Kolkata Knight Riders"),
    ["p_match", "year", "inns", "team_bat", "innings_score"],
].rename(
    columns={
        "inns": "opponent_innings_number",
        "team_bat": "opponent",
        "innings_score": "opponent_score",
    }
)

# Match-level KKR scorecard
kkr_scorecard = kkr_innings_complete.merge(
    opponent_innings,
    on=["p_match", "year"],
    how="inner",
    validate="one_to_one",
).merge(
    kkr_matches[["p_match", "year", "team_result"]],
    on=["p_match", "year"],
    how="inner",
    validate="one_to_one",
)

kkr_scorecard["kkr_batted_first"] = (
    kkr_scorecard["kkr_innings_number"].eq(1)
)

display(
    kkr_scorecard.sort_values(["year", "p_match"])
)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge

In [26]:
# Check whether KKR has more than one represented innings per match
kkr_duplicate_keys = (
    kkr_innings_complete
    .groupby(["p_match", "year"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

# Check whether the opponent innings table has multiple rows per match
opponent_duplicate_keys = (
    opponent_innings
    .groupby(["p_match", "year"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

print("Duplicate KKR match keys:", len(kkr_duplicate_keys))
print("Duplicate opponent match keys:", len(opponent_duplicate_keys))

display(opponent_duplicate_keys)

if not opponent_duplicate_keys.empty:
    display(
        opponent_innings.merge(
            opponent_duplicate_keys[["p_match", "year"]],
            on=["p_match", "year"],
            how="inner",
        ).sort_values(["year", "p_match", "opponent_innings_number"])
    )

Duplicate KKR match keys: 0
Duplicate opponent match keys: 116


,p_match,year,row_count
0,1359475,2023,2
2,1359477,2023,2
3,1359478,2023,2
4,1359479,2023,2
5,1359480,2023,2
...,...,...,...
137,1426305,2024,2
138,1426306,2024,2
139,1426307,2024,2
141,1426310,2024,2


,p_match,year,opponent_innings_number,opponent,opponent_score
0,1359475,2023,1,Chennai Super Kings,178
1,1359475,2023,2,Gujarat Titans,182
2,1359477,2023,1,Lucknow Super Giants,193
3,1359477,2023,2,Delhi Capitals,143
4,1359478,2023,1,Rajasthan Royals,203
...,...,...,...,...,...
227,1426307,2024,2,Sunrisers Hyderabad,215
228,1426310,2024,1,Royal Challengers Bengaluru,172
229,1426310,2024,2,Rajasthan Royals,174
230,1426311,2024,1,Sunrisers Hyderabad,175


In [27]:
# 1. Get KKR's innings from completed matches
kkr_innings_complete = complete_team_innings.loc[
    complete_team_innings["team_bat"].eq("Kolkata Knight Riders"),
    ["p_match", "year", "inns", "innings_score"],
].rename(
    columns={
        "inns": "kkr_innings_number",
        "innings_score": "kkr_score",
    }
)

# 2. Identify the match IDs in which KKR played
kkr_match_keys = kkr_innings_complete[["p_match", "year"]].drop_duplicates()

# 3. Select only the non-KKR innings from those same matches
opponent_innings = (
    complete_team_innings.merge(
        kkr_match_keys,
        on=["p_match", "year"],
        how="inner",
        validate="many_to_one",
    )
    .loc[
        lambda x: ~x["team_bat"].eq("Kolkata Knight Riders"),
        ["p_match", "year", "inns", "team_bat", "innings_score"],
    ]
    .rename(
        columns={
            "inns": "opponent_innings_number",
            "team_bat": "opponent",
            "innings_score": "opponent_score",
        }
    )
)

# 4. Build the match-level KKR scorecard
kkr_scorecard = (
    kkr_innings_complete.merge(
        opponent_innings,
        on=["p_match", "year"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        kkr_matches[["p_match", "year", "team_result"]],
        on=["p_match", "year"],
        how="inner",
        validate="one_to_one",
    )
)

# 5. Determine whether KKR batted first
kkr_scorecard["kkr_batted_first"] = (
    kkr_scorecard["kkr_innings_number"].eq(1)
)

print("KKR completed matches:", len(kkr_scorecard))
print("Duplicate match keys:", kkr_scorecard.duplicated(
    ["p_match", "year"]
).sum())

display(
    kkr_scorecard.sort_values(["year", "p_match"])
)

KKR completed matches: 28
Duplicate match keys: 0


,p_match,year,kkr_innings_number,kkr_score,opponent_innings_number,opponent,opponent_score,team_result,kkr_batted_first
0,1359476,2023,2,146,1,Punjab Kings,191,loss,False
1,1359483,2023,1,204,2,Royal Challengers Bengaluru,123,win,True
2,1359487,2023,2,207,1,Gujarat Titans,204,win,False
3,1359493,2023,2,204,1,Sunrisers Hyderabad,228,loss,False
4,1359496,2023,1,185,2,Mumbai Indians,186,loss,True
5,1359502,2023,1,127,2,Delhi Capitals,128,loss,True
6,1359507,2023,2,186,1,Chennai Super Kings,235,loss,False
7,1359510,2023,1,200,2,Royal Challengers Bengaluru,179,win,True
8,1359513,2023,1,179,2,Gujarat Titans,180,loss,True
9,1359521,2023,1,171,2,Sunrisers Hyderabad,166,win,True


In [28]:
kkr_batting_order_summary = (
    kkr_scorecard
    .groupby(
        ["year", "kkr_batted_first"],
        as_index=False,
        observed=True,
    )
    .agg(
        matches=("p_match", "nunique"),
        wins=("team_result", lambda s: s.eq("win").sum()),
        losses=("team_result", lambda s: s.eq("loss").sum()),
    )
)

kkr_batting_order_summary["win_pct"] = (
    100
    * kkr_batting_order_summary["wins"]
    / kkr_batting_order_summary["matches"]
)

kkr_batting_order_summary["batting_order"] = (
    kkr_batting_order_summary["kkr_batted_first"]
    .map({
        True: "Batted first",
        False: "Chased",
    })
)

kkr_batting_order_summary = kkr_batting_order_summary[
    [
        "year",
        "batting_order",
        "matches",
        "wins",
        "losses",
        "win_pct",
    ]
].sort_values(["year", "batting_order"])

display(kkr_batting_order_summary.round({"win_pct": 2}))

,year,batting_order,matches,wins,losses,win_pct
1,2023,Batted first,7,3,4,42.86
0,2023,Chased,7,3,4,42.86
3,2024,Batted first,9,6,3,66.67
2,2024,Chased,5,5,0,100.00


In [29]:
kkr_batting_order_scores = (
    kkr_scorecard
    .groupby(
        ["year", "kkr_batted_first"],
        as_index=False,
        observed=True,
    )
    .agg(
        matches=("p_match", "nunique"),
        mean_kkr_score=("kkr_score", "mean"),
        mean_opponent_score=("opponent_score", "mean"),
        mean_score_difference=(
            "kkr_score",
            lambda s: (
                s - kkr_scorecard.loc[s.index, "opponent_score"]
            ).mean(),
        ),
    )
)

kkr_batting_order_scores["batting_order"] = (
    kkr_batting_order_scores["kkr_batted_first"]
    .map({
        True: "Batted first",
        False: "Chased",
    })
)

display(
    kkr_batting_order_scores[
        [
            "year",
            "batting_order",
            "matches",
            "mean_kkr_score",
            "mean_opponent_score",
            "mean_score_difference",
        ]
    ].round(2)
)

,year,batting_order,matches,mean_kkr_score,mean_opponent_score,mean_score_difference
0,2023,Chased,7,178.14,193.86,-15.71
1,2023,Batted first,7,173.57,159.00,14.57
2,2024,Chased,5,156.40,153.60,2.80
3,2024,Batted first,9,208.89,181.22,27.67


In [30]:
kkr_successful_chases_2024 = (
    kkr_scorecard.loc[
        kkr_scorecard["year"].eq(2024)
        & ~kkr_scorecard["kkr_batted_first"]
        & kkr_scorecard["team_result"].eq("win"),
        [
            "p_match",
            "opponent",
            "opponent_score",
            "kkr_score",
            "team_result",
        ],
    ]
    .copy()
    .sort_values("opponent_score", ascending=False)
)

kkr_successful_chases_2024["runs_above_target"] = (
    kkr_successful_chases_2024["kkr_score"]
    - kkr_successful_chases_2024["opponent_score"]
)

display(kkr_successful_chases_2024)

,p_match,opponent,opponent_score,kkr_score,team_result,runs_above_target
15,1422128,Royal Challengers Bengaluru,182,186,win,4
18,1426266,Lucknow Super Giants,161,162,win,1
26,1426309,Sunrisers Hyderabad,159,164,win,5
22,1426285,Delhi Capitals,153,156,win,3
27,1426312,Sunrisers Hyderabad,113,114,win,1


In [31]:
kkr_deliveries = df_features.loc[
    df_features["team_bat"].eq("Kolkata Knight Riders")
].copy()

kkr_phase_summary = (
    kkr_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        batting_runs=("batruns", "sum"),
        balls_faced=("ballfaced", "sum"),
        batting_dots=("batting_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs=("boundary_runs", "sum"),
        innings_count=("p_match", lambda s: s.nunique()),
    )
    .reset_index()
)

kkr_phase_summary["strike_rate"] = (
    100
    * kkr_phase_summary["batting_runs"]
    / kkr_phase_summary["balls_faced"].replace(0, pd.NA)
)

kkr_phase_summary["dot_ball_pct"] = (
    100
    * kkr_phase_summary["batting_dots"]
    / kkr_phase_summary["balls_faced"].replace(0, pd.NA)
)

kkr_phase_summary["boundary_ball_pct"] = (
    100
    * kkr_phase_summary["boundary_balls"]
    / kkr_phase_summary["balls_faced"].replace(0, pd.NA)
)

kkr_phase_summary["boundary_run_pct"] = (
    100
    * kkr_phase_summary["boundary_runs"]
    / kkr_phase_summary["batting_runs"].replace(0, pd.NA)
)

phase_order = ["Powerplay", "Middle", "Death"]

kkr_phase_summary["match_phase"] = pd.Categorical(
    kkr_phase_summary["match_phase"],
    categories=phase_order,
    ordered=True,
)

display(
    kkr_phase_summary.sort_values(["year", "match_phase"]).round(2)
)

,year,match_phase,batting_runs,balls_faced,batting_dots,boundary_balls,boundary_runs,innings_count,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
2,2023,Powerplay,651,507,264,109,494,14,128.40,52.07,21.50,75.88
1,2023,Middle,1065,760,233,130,626,14,140.13,30.66,17.11,58.78
0,2023,Death,622,390,146,88,438,14,159.49,37.44,22.56,70.42
5,2024,Powerplay,849,507,231,153,702,14,167.46,45.56,30.18,82.69
4,2024,Middle,1114,725,224,149,712,14,153.66,30.90,20.55,63.91
3,2024,Death,518,275,83,77,384,12,188.36,30.18,28.00,74.13


In [32]:
kkr_phase_coverage = (
    kkr_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        represented_matches=("p_match", "nunique"),
        represented_innings=("p_match", lambda s: (
            kkr_deliveries.loc[s.index, ["p_match", "inns"]]
            .drop_duplicates()
            .shape[0]
        )),
        delivery_records=("p_match", "size"),
        legal_balls=("legal_delivery", "sum"),
        balls_faced=("ballfaced", "sum"),
    )
    .reset_index()
)

kkr_phase_coverage["match_phase"] = pd.Categorical(
    kkr_phase_coverage["match_phase"],
    categories=["Powerplay", "Middle", "Death"],
    ordered=True,
)

display(
    kkr_phase_coverage
    .sort_values(["year", "match_phase"])
)

,year,match_phase,represented_matches,represented_innings,delivery_records,legal_balls,balls_faced
2,2023,Powerplay,14,14,529,504,507
1,2023,Middle,14,14,781,756,760
0,2023,Death,14,14,418,387,390
5,2024,Powerplay,14,14,531,504,507
4,2024,Middle,14,14,759,721,725
3,2024,Death,12,12,304,269,275


In [33]:
# Distinct KKR innings represented in the Death phase
death_innings = (
    kkr_deliveries.loc[
        kkr_deliveries["match_phase"].eq("Death")
    ]
    [["year", "p_match", "inns"]]
    .drop_duplicates()
)

# All represented KKR innings
all_kkr_innings = (
    kkr_deliveries
    [["year", "p_match", "inns"]]
    .drop_duplicates()
)

# Identify innings with no Death-phase records
innings_without_death = (
    all_kkr_innings
    .merge(
        death_innings.assign(has_death_phase=True),
        on=["year", "p_match", "inns"],
        how="left",
        validate="one_to_one",
    )
    .loc[lambda x: x["has_death_phase"].isna()]
    .drop(columns="has_death_phase")
)

display(innings_without_death.sort_values(["year", "p_match"]))

,year,p_match,inns
26,2024,1426309,2
27,2024,1426312,2


In [34]:
# Inspect the full innings context for the identified cases
innings_context = (
    kkr_deliveries
    .groupby(["year", "p_match", "inns"], as_index=False)
    .agg(
        innings_score=("score", "sum"),
        last_over=("over_num", "max"),
        delivery_records=("score", "size"),
        legal_balls=("legal_delivery", "sum"),
        balls_faced=("ballfaced", "sum"),
    )
)

display(
    innings_context.merge(
        innings_without_death,
        on=["year", "p_match", "inns"],
        how="inner",
        validate="one_to_one",
    )
)

,year,p_match,inns,innings_score,last_over,delivery_records,legal_balls,balls_faced
0,2024,1426309,2,164,14,84,82,82
1,2024,1426312,2,114,11,67,63,63


In [35]:
kkr_powerplay_batters = (
    kkr_deliveries.loc[
        kkr_deliveries["match_phase"].eq("Powerplay")
    ]
    .groupby(["year", "p_bat", "bat"], as_index=False)
    .agg(
        runs=("batruns", "sum"),
        balls_faced=("ballfaced", "sum"),
        dot_balls=("batting_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        matches=("p_match", "nunique"),
    )
)

kkr_powerplay_batters["strike_rate"] = (
    100
    * kkr_powerplay_batters["runs"]
    / kkr_powerplay_batters["balls_faced"].replace(0, pd.NA)
)

kkr_powerplay_batters["dot_ball_pct"] = (
    100
    * kkr_powerplay_batters["dot_balls"]
    / kkr_powerplay_batters["balls_faced"].replace(0, pd.NA)
)

kkr_powerplay_batters["boundary_ball_pct"] = (
    100
    * kkr_powerplay_batters["boundary_balls"]
    / kkr_powerplay_batters["balls_faced"].replace(0, pd.NA)
)

display(
    kkr_powerplay_batters
    .sort_values(["year", "runs"], ascending=[True, False])
    .round(2)
)

,year,p_bat,bat,runs,balls_faced,dot_balls,boundary_balls,matches,strike_rate,dot_ball_pct,boundary_ball_pct
1,2023,298438,Jason Roy,177,121,59,33,7,146.28,48.76,27.27
8,2023,974087,Rahmanullah Gurbaz,145,112,60,22,10,129.46,53.57,19.64
7,2023,851403,Venkatesh Iyer,142,109,55,23,12,130.28,50.46,21.10
5,2023,604527,Nitish Rana,94,71,35,15,12,132.39,49.30,21.13
9,2023,1048813,Narayan Jagadeesan,63,58,32,11,6,108.62,55.17,18.97
6,2023,723105,Rinku Singh,18,11,4,3,2,163.64,36.36,27.27
2,2023,398506,Mandeep Singh,4,8,4,0,3,50.00,50.00,0.00
4,2023,536936,Litton Das,4,4,3,1,1,100.00,75.00,25.00
10,2023,1079839,Anukul Roy,4,5,4,1,1,80.00,80.00,20.00
0,2023,230558,Sunil Narine,0,4,4,0,2,0.00,100.00,0.00


In [36]:
# Total KKR Powerplay runs by season
kkr_powerplay_totals = (
    kkr_powerplay_batters
    .groupby("year", as_index=False)
    .agg(
        team_powerplay_runs=("runs", "sum"),
        team_powerplay_balls=("balls_faced", "sum"),
    )
)

# Calculate each batter's share of team Powerplay runs
kkr_powerplay_contributions = kkr_powerplay_batters.merge(
    kkr_powerplay_totals,
    on="year",
    how="left",
    validate="many_to_one",
)

kkr_powerplay_contributions["run_share_pct"] = (
    100
    * kkr_powerplay_contributions["runs"]
    / kkr_powerplay_contributions["team_powerplay_runs"]
)

# Identify whether the player appeared in both seasons
player_season_presence = (
    kkr_powerplay_batters
    .groupby("p_bat")["year"]
    .nunique()
    .rename("seasons_present")
    .reset_index()
)

kkr_powerplay_contributions = kkr_powerplay_contributions.merge(
    player_season_presence,
    on="p_bat",
    how="left",
    validate="many_to_one",
)

kkr_powerplay_contributions["presence_category"] = (
    kkr_powerplay_contributions["seasons_present"]
    .map({
        1: "One season only",
        2: "Both seasons",
    })
)

display(
    kkr_powerplay_contributions[
        [
            "year",
            "bat",
            "runs",
            "balls_faced",
            "strike_rate",
            "run_share_pct",
            "presence_category",
        ]
    ]
    .sort_values(["year", "runs"], ascending=[True, False])
    .round(2)
)

,year,bat,runs,balls_faced,strike_rate,run_share_pct,presence_category
1,2023,Jason Roy,177,121,146.28,27.19,One season only
8,2023,Rahmanullah Gurbaz,145,112,129.46,22.27,Both seasons
7,2023,Venkatesh Iyer,142,109,130.28,21.81,Both seasons
5,2023,Nitish Rana,94,71,132.39,14.44,Both seasons
9,2023,Narayan Jagadeesan,63,58,108.62,9.68,One season only
6,2023,Rinku Singh,18,11,163.64,2.76,Both seasons
2,2023,Mandeep Singh,4,8,50.00,0.61,One season only
4,2023,Litton Das,4,4,100.00,0.61,One season only
10,2023,Anukul Roy,4,5,80.00,0.61,One season only
0,2023,Sunil Narine,0,4,0.00,0.00,Both seasons


In [37]:
# Keep only players represented in both seasons
returning_batters = kkr_powerplay_contributions.loc[
    kkr_powerplay_contributions["presence_category"].eq("Both seasons")
].copy()

returning_batter_comparison = returning_batters.pivot(
    index="bat",
    columns="year",
    values=["runs", "balls_faced", "strike_rate", "run_share_pct"],
)

# Flatten the multi-level column names
returning_batter_comparison.columns = [
    f"{metric}_{year}"
    for metric, year in returning_batter_comparison.columns
]

returning_batter_comparison = returning_batter_comparison.reset_index()

# Calculate changes where both seasons have observations
returning_batter_comparison["runs_change"] = (
    returning_batter_comparison["runs_2024"]
    - returning_batter_comparison["runs_2023"]
)

returning_batter_comparison["balls_change"] = (
    returning_batter_comparison["balls_faced_2024"]
    - returning_batter_comparison["balls_faced_2023"]
)

returning_batter_comparison["strike_rate_change"] = (
    returning_batter_comparison["strike_rate_2024"]
    - returning_batter_comparison["strike_rate_2023"]
)

display(
    returning_batter_comparison.sort_values(
        "runs_change",
        ascending=False,
    ).round(2)
)

,bat,runs_2023,runs_2024,balls_faced_2023,balls_faced_2024,strike_rate_2023,strike_rate_2024,run_share_pct_2023,run_share_pct_2024,runs_change,balls_change,strike_rate_change
3,Sunil Narine,0.0,281.0,4.0,165.0,0.00,170.30,0.00,33.10,281.0,161.0,170.30
2,Rinku Singh,18.0,9.0,11.0,7.0,163.64,128.57,2.76,1.06,-9.0,-4.0,-35.06
4,Venkatesh Iyer,142.0,111.0,109.0,50.0,130.28,222.00,21.81,13.07,-31.0,-59.0,91.72
0,Nitish Rana,94.0,12.0,71.0,14.0,132.39,85.71,14.44,1.41,-82.0,-57.0,-46.68
1,Rahmanullah Gurbaz,145.0,44.0,112.0,36.0,129.46,122.22,22.27,5.18,-101.0,-76.0,-7.24


In [38]:
# Filter deliveries where KKR was the bowling team
kkr_bowling_deliveries = df_features.loc[
    df_features["team_bowl"].eq("Kolkata Knight Riders")
].copy()

# Aggregate bowling performance by season and phase
kkr_bowling_phase = (
    kkr_bowling_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        bowler_credited_wickets=("bowler_credited_wicket", "sum"),
        delivery_records=("p_match", "size"),
        represented_matches=("p_match", "nunique"),
    )
    .reset_index()
)

# Convert legal balls into overs and calculate bowling rates
kkr_bowling_phase["overs_bowled"] = (
    kkr_bowling_phase["legal_balls"] / 6
)

kkr_bowling_phase["runs_per_over"] = (
    kkr_bowling_phase["runs_conceded"]
    / kkr_bowling_phase["overs_bowled"].replace(0, pd.NA)
)

kkr_bowling_phase["dot_ball_pct"] = (
    100
    * kkr_bowling_phase["dot_balls"]
    / kkr_bowling_phase["legal_balls"].replace(0, pd.NA)
)

kkr_bowling_phase["wickets_per_20_overs"] = (
    20
    * kkr_bowling_phase["bowler_credited_wickets"]
    / kkr_bowling_phase["overs_bowled"].replace(0, pd.NA)
)

# Sort phases in cricket order
phase_order = ["Powerplay", "Middle", "Death"]

kkr_bowling_phase["match_phase"] = pd.Categorical(
    kkr_bowling_phase["match_phase"],
    categories=phase_order,
    ordered=True,
)

display(
    kkr_bowling_phase
    .sort_values(["year", "match_phase"])
    .round(2)
)

,year,match_phase,runs_conceded,legal_balls,dot_balls,bowler_credited_wickets,delivery_records,represented_matches,overs_bowled,runs_per_over,dot_ball_pct,wickets_per_20_overs
2,2023,Powerplay,806,504,198,20,525,14,84.00,9.60,39.29,4.76
1,2023,Middle,1009,745,247,36,758,14,124.17,8.13,33.15,5.80
0,2023,Death,609,345,86,21,374,13,57.50,10.59,24.93,7.30
5,2024,Powerplay,817,504,211,27,540,14,84.00,9.73,41.87,6.43
4,2024,Middle,1022,756,265,49,783,14,126.00,8.11,35.05,7.78
3,2024,Death,517,316,121,28,328,14,52.67,9.82,38.29,10.63


In [39]:
kkr_bowling_coverage = (
    kkr_bowling_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        represented_matches=("p_match", "nunique"),
        represented_innings=("p_match", lambda s: (
            kkr_bowling_deliveries.loc[
                s.index, ["p_match", "inns"]
            ]
            .drop_duplicates()
            .shape[0]
        )),
        delivery_records=("p_match", "size"),
        legal_balls=("legal_delivery", "sum"),
    )
    .reset_index()
)

kkr_bowling_coverage["match_phase"] = pd.Categorical(
    kkr_bowling_coverage["match_phase"],
    categories=["Powerplay", "Middle", "Death"],
    ordered=True,
)

display(
    kkr_bowling_coverage
    .sort_values(["year", "match_phase"])
)

,year,match_phase,represented_matches,represented_innings,delivery_records,legal_balls
2,2023,Powerplay,14,14,525,504
1,2023,Middle,14,14,758,745
0,2023,Death,13,13,374,345
5,2024,Powerplay,14,14,540,504
4,2024,Middle,14,14,783,756
3,2024,Death,14,14,328,316


In [40]:
kkr_death_bowlers = (
    kkr_bowling_deliveries.loc[
        kkr_bowling_deliveries["match_phase"].eq("Death")
    ]
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        matches=("p_match", "nunique"),
    )
)

kkr_death_bowlers["overs_bowled"] = (
    kkr_death_bowlers["legal_balls"] / 6
)

kkr_death_bowlers["runs_per_over"] = (
    kkr_death_bowlers["runs_conceded"]
    / kkr_death_bowlers["overs_bowled"].replace(0, pd.NA)
)

kkr_death_bowlers["dot_ball_pct"] = (
    100
    * kkr_death_bowlers["dot_balls"]
    / kkr_death_bowlers["legal_balls"].replace(0, pd.NA)
)

kkr_death_bowlers["wickets_per_20_overs"] = (
    20
    * kkr_death_bowlers["wickets"]
    / kkr_death_bowlers["overs_bowled"].replace(0, pd.NA)
)

display(
    kkr_death_bowlers
    .sort_values(["year", "wickets"], ascending=[True, False])
    .round(2)
)

,year,p_bowl,bowl,runs_conceded,legal_balls,dot_balls,wickets,matches,overs_bowled,runs_per_over,dot_ball_pct,wickets_per_20_overs
9,2023,1108375,Varun Chakravarthy,106,76,24,4,11,12.67,8.37,31.58,6.32
0,2023,230558,Sunil Narine,69,48,12,3,8,8.00,8.62,25.00,7.50
4,2023,475281,Shardul Thakur,88,47,11,3,6,7.83,11.23,23.40,7.66
10,2023,1209292,Vaibhav Arora,67,36,8,3,5,6.00,11.17,22.22,10.00
2,2023,276298,Andre Russell,60,31,8,2,5,5.17,11.61,25.81,7.74
8,2023,1083033,Kulwant Khejroliya,37,14,4,2,2,2.33,15.86,28.57,17.14
5,2023,493773,Lockie Ferguson,37,16,1,1,2,2.67,13.88,6.25,7.50
6,2023,604527,Nitish Rana,22,17,4,1,2,2.83,7.76,23.53,7.06
7,2023,1079839,Anukul Roy,6,6,2,1,1,1.00,6.00,33.33,20.00
12,2023,1350792,Suyash Sharma,68,36,9,1,6,6.00,11.33,25.00,3.33


In [41]:
death_bowling_contribution = kkr_death_bowlers.copy()

# Team totals for each season
death_team_totals = (
    death_bowling_contribution
    .groupby("year", as_index=False)
    .agg(
        team_legal_balls=("legal_balls", "sum"),
        team_wickets=("wickets", "sum"),
        team_runs_conceded=("runs_conceded", "sum"),
    )
)

death_bowling_contribution = death_bowling_contribution.merge(
    death_team_totals,
    on="year",
    how="left",
    validate="many_to_one",
)

death_bowling_contribution["legal_ball_share_pct"] = (
    100
    * death_bowling_contribution["legal_balls"]
    / death_bowling_contribution["team_legal_balls"]
)

death_bowling_contribution["wicket_share_pct"] = (
    100
    * death_bowling_contribution["wickets"]
    / death_bowling_contribution["team_wickets"]
)

display(
    death_bowling_contribution[
        [
            "year",
            "bowl",
            "matches",
            "legal_balls",
            "legal_ball_share_pct",
            "runs_per_over",
            "dot_ball_pct",
            "wickets",
            "wicket_share_pct",
        ]
    ]
    .sort_values(
        ["year", "legal_balls"],
        ascending=[True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

,year,bowl,matches,legal_balls,legal_ball_share_pct,runs_per_over,dot_ball_pct,wickets,wicket_share_pct
0,2023,Varun Chakravarthy,11,76,22.03,8.37,31.58,4,19.05
1,2023,Sunil Narine,8,48,13.91,8.62,25.00,3,14.29
2,2023,Shardul Thakur,6,47,13.62,11.23,23.40,3,14.29
3,2023,Vaibhav Arora,5,36,10.43,11.17,22.22,3,14.29
4,2023,Suyash Sharma,6,36,10.43,11.33,25.00,1,4.76
5,2023,Andre Russell,5,31,8.99,11.61,25.81,2,9.52
6,2023,Nitish Rana,2,17,4.93,7.76,23.53,1,4.76
7,2023,Lockie Ferguson,2,16,4.64,13.88,6.25,1,4.76
8,2023,Kulwant Khejroliya,2,14,4.06,15.86,28.57,2,9.52
9,2023,Tim Southee,1,6,1.74,15.00,16.67,0,0.00


In [42]:
kkr_middle_bowlers = (
    kkr_bowling_deliveries.loc[
        kkr_bowling_deliveries["match_phase"].eq("Middle")
    ]
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        matches=("p_match", "nunique"),
    )
)

kkr_middle_bowlers["overs_bowled"] = (
    kkr_middle_bowlers["legal_balls"] / 6
)

kkr_middle_bowlers["runs_per_over"] = (
    kkr_middle_bowlers["runs_conceded"]
    / kkr_middle_bowlers["overs_bowled"].replace(0, pd.NA)
)

kkr_middle_bowlers["dot_ball_pct"] = (
    100
    * kkr_middle_bowlers["dot_balls"]
    / kkr_middle_bowlers["legal_balls"].replace(0, pd.NA)
)

kkr_middle_bowlers["wickets_per_20_overs"] = (
    20
    * kkr_middle_bowlers["wickets"]
    / kkr_middle_bowlers["overs_bowled"].replace(0, pd.NA)
)

display(
    kkr_middle_bowlers
    .sort_values(
        ["year", "legal_balls"],
        ascending=[True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

,year,p_bowl,bowl,runs_conceded,legal_balls,dot_balls,wickets,matches,overs_bowled,runs_per_over,dot_ball_pct,wickets_per_20_overs
0,2023,230558,Sunil Narine,245,198,52,6,14,33.00,7.42,26.26,3.64
1,2023,1350792,Suyash Sharma,231,180,64,6,11,30.00,7.70,35.56,4.00
2,2023,1108375,Varun Chakravarthy,190,144,52,11,14,24.00,7.92,36.11,9.17
3,2023,475281,Shardul Thakur,82,55,17,3,7,9.17,8.95,30.91,6.55
4,2023,1079839,Anukul Roy,59,42,18,2,3,7.00,8.43,42.86,5.71
5,2023,604527,Nitish Rana,27,30,14,2,4,5.00,5.40,46.67,8.00
6,2023,276298,Andre Russell,39,24,8,1,3,4.00,9.75,33.33,5.00
7,2023,376116,Umesh Yadav,53,24,4,1,3,4.00,13.25,16.67,5.00
8,2023,493773,Lockie Ferguson,39,18,8,0,3,3.00,13.00,44.44,0.00
9,2023,1312645,Harshit Rana,15,12,3,1,2,2.00,7.50,25.00,10.00


In [43]:
from pathlib import Path

metric_file = project_root / "src" / "features" / "metric_definitions.py"
test_file = project_root / "tests" / "test_metric_definitions.py"

print("=== metric_definitions.py ===")
print(metric_file.read_text())

print("\n=== test_metric_definitions.py ===")
print(test_file.read_text())

=== metric_definitions.py ===
"""Reusable metric definitions for delivery-level IPL analytics.

The functions in this module derive analytical features without modifying
the input DataFrame.

Important:
- The raw dataset must remain unchanged.
- Batting dot balls remain a provisional definition.
- Source bowlruns anomalies must not be silently corrected.
"""

from __future__ import annotations
from unittest import result

import pandas as pd


# Dismissals included in the candidate bowler-credited wicket metric.
BOWLER_CREDITED_DISMISSALS = {
    "caught",
    "bowled",
    "leg before wicket",
    "stumped",
    "hit wicket",
}


def derive_legal_delivery(df: pd.DataFrame) -> pd.Series:
    """Return whether each delivery is legal.

    A delivery is legal when neither a wide nor a no-ball is recorded.
    """
    required_columns = {"wide", "noball"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing colum

In [44]:
from pathlib import Path

player_level_file = project_root / "src" / "features" / "player_level.py"

print(player_level_file.read_text())

"""Build player-level analytical datasets from delivery records."""

from __future__ import annotations

import pandas as pd


# Grouping keys defining each analytical dataset.
PLAYER_SEASON_BATTING_KEYS = [
    "p_bat",
    "year",
]

PLAYER_SEASON_BOWLING_KEYS = [
    "p_bowl",
    "year",
]

PLAYER_SEASON_PHASE_BATTING_KEYS = [
    "p_bat",
    "year",
    "match_phase",
]

PLAYER_SEASON_PHASE_BOWLING_KEYS = [
    "p_bowl",
    "year",
    "match_phase",
]


def build_player_season_batting(
        
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Build one row per batter per season."""

    required_columns = {
        "p_bat",
        "bat",
        "year",
        "batruns",
        "ballfaced",
        "batting_dot",
        "boundary_ball",
        "boundary_runs",
    }

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Aggregate the unde